## Setup


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from isabelle_connector.isabelle_connector import IsabelleConnector
from isabelle_connector.utils import get_theory, list_theory_files, temp_theory

In [3]:
isabelle = IsabelleConnector(
    name="test_connector", working_directory=".", debug=True
)

## Hello, World! Test Theory


In [4]:
test_thy = temp_theory(
    name="ConnectorTest",
    working_directory=isabelle.working_directory,
    imports=[],
    queries=[
        'ML\\<open> let val res = "Hello, World!" in res end \\<close>',  # Make sure to define any values you want to retrieve in the ML block
    ],
)

In [5]:
test_thy_output, test_thy_errs = isabelle.use_theories([test_thy], use_cache=False)

Using cached results for 0 / 1 theories
Starting session 1 / 1: HOL


DONE:   0%|          | 0/1 [00:00<?, ?it/s]

Successful values from 1 / 1 theories
func:use_theories took: 9.387954711914062 sec


In [6]:
test_thy_output.values()

dict_values([['Hello, World!']])

## Trivial Theorem Test Theory


In [7]:
trivial_theorem_thy = temp_theory(
    name="TrivialTheorem",
    working_directory=isabelle.working_directory,
    imports=[],
    queries=[
        "theorem TrueI: True",
        "sorry",
        'ML\\<open> val thm = @{thm "TrueI"} \\<close>',
    ],
)

In [8]:
print(trivial_theorem_thy)

theory TrivialTheorem
            imports Main  begin
            declare [[show_markup = false]]
            declare [[show_consts = true]]
            declare [[show_abbrevs = true]]
            declare [[names_long = false]]
            declare [[ML_print_depth=1000000]]
            declare [[syntax_ambiguity_warning = false]]
            theorem TrueI: True
sorry
ML\<open> val thm = @{thm "TrueI"} \<close>
            end


In [9]:
trivial_theorem_thy_output, trivial_theorem_thy_errs = isabelle.use_theories([trivial_theorem_thy], rm_after=False) # keep the temporary theory file to retrieve transitions next

Using cached results for 1 / 1 theories
Successful values from 1 / 1 theories
func:use_theories took: 0.0013477802276611328 sec


In [10]:
trivial_theorem_thy_output.values()

dict_values([['True']])

## Transitions of Trivial Theorem


In [11]:
from argparse import Namespace

from isabelle_connector.data_extraction import transitions_theory

from isabelle_connector.config import PROJ_ROOT, HOL_DIR


transition_configs = Namespace(
    imports=[PROJ_ROOT / "isabelle-thys/ExtractLemmas", PROJ_ROOT / "isabelle-thys/RoughSpec"],
    root_dir=HOL_DIR,
)
transitions_of_trivial_thy = transitions_theory(trivial_theorem_thy, transition_configs)

In [14]:
transitions_of_trivial_output, transitions_of_trivial_errs = isabelle.use_theories([transitions_of_trivial_thy])

Using cached results for 1 / 1 theories
Successful values from 1 / 1 theories
func:use_theories took: 0.0020177364349365234 sec


In [15]:
transitions_of_trivial_output.values()

dict_values([[('TrivialTheorem', [('theory', 'theory TrivialTheorem\n            imports Main  begin'), ('<ignored>', '\n            '), ('declare', 'declare [[show_markup = False]]'), ('<ignored>', '\n            '), ('declare', 'declare [[show_consts = True]]'), ('<ignored>', '\n            '), ('declare', 'declare [[show_abbrevs = True]]'), ('<ignored>', '\n            '), ('declare', 'declare [[names_long = False]]'), ('<ignored>', '\n            '), ('declare', 'declare [[ML_print_depth=1000000]]'), ('<ignored>', '\n            '), ('declare', 'declare [[syntax_ambiguity_warning = False]]'), ('<ignored>', '\n            '), ('theorem', 'theorem TrueI: True'), ('<ignored>', '\n'), ('sorry', 'sorry'), ('<ignored>', '\n'), ('ML', 'ML\\<open> val thm = @{thm "TrueI"} \\<close>'), ('<ignored>', '\n            '), ('end', 'end')])]])

## Transitions of existing theory


In [12]:
theory_name = "IMP/AExp.thy"
theory_object = get_theory(theory_name, HOL_DIR)
transitions_thy = transitions_theory(theory_object, transition_configs)

In [13]:
transitions_output, transitions_errs = isabelle.use_theories([transitions_thy], use_cache=False)

Using cached results for 0 / 1 theories


DONE:   0%|          | 0/1 [00:00<?, ?it/s]

Successful values from 1 / 1 theories
func:use_theories took: 2.4304096698760986 sec


In [28]:
transitions_output.values()

dict_values([[('IMP/AExp', [('section', 'section "Arithmetic and Boolean Expressions"'), ('<ignored>', '\n\n'), ('subsection', 'subsection "Arithmetic Expressions"'), ('<ignored>', '\n\n'), ('theory', 'theory AExp imports Main begin'), ('<ignored>', '\n\n'), ('type_synonym', 'type_synonym vname = string'), ('<ignored>', '\n'), ('type_synonym', 'type_synonym val = int'), ('<ignored>', '\n'), ('type_synonym', 'type_synonym state = "vname \\<Rightarrow> val"'), ('<ignored>', '\n\n'), ('text_raw', 'text_raw\\<open>\\snip{AExpaexpdef}{2}{1}{%\\<close>'), ('<ignored>', '\n'), ('datatype', 'datatype aexp = N int | V vname | Plus aexp aexp'), ('<ignored>', '\n'), ('text_raw', 'text_raw\\<open>}%endsnip\\<close>'), ('<ignored>', '\n\n'), ('text_raw', 'text_raw\\<open>\\snip{AExpavaldef}{1}{2}{%\\<close>'), ('<ignored>', '\n'), ('fun', 'fun aval :: "aexp \\<Rightarrow> state \\<Rightarrow> val" where\n"aval (N n) s = n" |\n"aval (V x) s = s x" |\n"aval (Plus a\\<^sub>1 a\\<^sub>2) s = aval a\\<^

In [29]:
transitions_errs.values()

dict_values([[]])